# 03 - Synthetic vs real training data

Does LLM-generated training data match real MASSIVE data for a new intent? Two chains over the same 5 reserve intents: synthetic train with real eval/test, and fully synthetic splits for the dual (synthetic vs real test) evaluation.

In [ ]:
import os
from google.colab import drive, userdata
drive.mount('/content/drive')
os.environ['WANDB_API_KEY'] = userdata.get('wandbKey')
os.environ['ANTHROPIC_API_KEY'] = userdata.get('anthropic_key')

In [ ]:
%pip install -q "transformers>=4.48,<5" "huggingface_hub>=0.34,<1" "datasets<3.0.0" anthropic

In [ ]:
%cd /content/drive/MyDrive/incremental_learning_on_the_edge/src
!git pull

In [ ]:
from core.configuration import load_config
from core.dataloader import DataClass
from core.evaluate import intents_report, old_intent_persistance_table, new_intent_acquisition_table
from experiments.runners import (
    run_synthetic_incremental_experiment,
    run_production_incremental_experiment,
)

config = load_config()
INTENTS = ["takeaway_order", "general_joke", "recommendation_locations", "play_podcasts", "transport_traffic"]

### A. Synthetic training data, real eval/test

In [ ]:
dataclass = run_synthetic_incremental_experiment(
    intents_to_add=INTENTS,
    student_key="student1",
    config=config,
    K=70,
    n_generate=130,
    seed=42,
)

### B. Fully synthetic splits (for the dual test evaluation)

In [ ]:
dataclass = run_production_incremental_experiment(
    intents_to_add=INTENTS,
    student_key="student1",
    config=config,
    K=70,
    n_generate=130,   # more than the split needs, in case of duplicates
    n_eval=20,
    n_test=20,
    seed=42,
)

### Evaluation: same models scored on the synthetic and on the real test split

In [ ]:
rows_syn, m_syn = intents_report(dataclass, "student1", config, n_versions=5, test_source="pretraining")
rows_real, m_real = intents_report(dataclass, "student1", config, n_versions=5, test_source="real")

print("retention (old intents)")
print(old_intent_persistance_table(rows_syn, dataclass.id2intent))
print("\nacquisition, synthetic test")
print(new_intent_acquisition_table(rows_syn, dataclass.id2intent))
print("\nacquisition, real test")
print(new_intent_acquisition_table(rows_real, dataclass.id2intent))

### Keep the generated dataset (so a run can be re-used byte-identically)

In [ ]:
dataclass.dataset_pretraining.save_to_disk("outputs/datasets/SYNTH_K70_g130_a0.2_s42")